[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 困难：GPT-2 Transformer Block

实现一个完整的 **GPT-2 风格的 Transformer Block** —— 结合你之前学到的所有知识。

### 架构（Pre-Norm）
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### 函数签名
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### 要求
- 继承自 `nn.Module`
- `self.ln1`、`self.ln2`：`nn.LayerNorm(d_model)`
- `self.W_q`、`self.W_k`、`self.W_v`、`self.W_o`：用于注意力机制的 `nn.Linear`
- `self.mlp`：`nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- 注意力必须是 **因果注意力**（causal，屏蔽未来位置）
- 采用 **Pre-Norm** 架构（LayerNorm 放在 Attention 和 MLP 之前）
- 在 Attention 和 MLP 周围都使用残差连接（Residual Connection）

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F

In [ ]:
# ✏️ 在这里实现你的代码

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        pass  # 初始化各个层

    def forward(self, x):
        pass  # Pre-Norm + 因果自注意力 + MLP + 残差连接

In [ ]:
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"

        # 1. 层归一化 (Pre-Norm 架构)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # 2. 注意力机制层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        # 3. MLP (Feed-Forward Network)
        # GPT-2 的标准配置是中间层维度为 4 * d_model，激活函数为 GELU/Relu
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(approximate='tanh'),
            nn.Linear(4 * d_model, d_model)
        )

    def causal_self_attention(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape

        # 线性投影并切分为多头
        # (B, T, D) -> (B, T, num_heads, head_dim) -> (B, num_heads, T, head_dim)
        q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # 缩放点积注意力 (Scaled Dot-Product Attention)
        # scores: (B, num_heads, T, T)
        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # 应用因果掩码 (Causal Mask)
        # 创建下三角矩阵，0 表示可见，-inf 表示屏蔽
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).view(1, 1, seq_len, seq_len)
        attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        # Softmax 与权重求和
        attn_weights = F.softmax(attn_scores, dim=-1)
        # (B, num_heads, T, head_dim)
        out = attn_weights @ v

        # 合并多头并映射回 d_model 维度
        # (B, T, D)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 第一部分：Pre-Norm -> Attention -> Residual Connection
        # x = x + causal_self_attention(ln1(x))
        x = x + self.causal_self_attention(self.ln1(x))

        # 第二部分：Pre-Norm -> MLP -> Residual Connection
        # x = x + mlp(ln2(x))
        x = x + self.mlp(self.ln2(x))

        return x

In [ ]:
# 🧪 调试测试
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("输出形状：", out.shape)           # 应为 (2, 8, 64)
print("是否为 nn.Module？", isinstance(block, nn.Module))
print("参数量：", sum(p.numel() for p in block.parameters()))

In [ ]:
from torch_judge import check
check('gpt2_block')